# ZuCo / TeCo text-only sentiment baseline

Frozen and fine-tuned baselines with three heads (**mean**, **cls**, and the
**lstm** text component from Hollenstein et al. 2021), on a held-out test set
via nested cross-validation. Runs on **ZuCo** (English) and **TeCo** (Persian).

Run top to bottom on a **GPU runtime** (*Runtime -> Change runtime type -> GPU*).

## 1. Get the code

In [ ]:
REPO="https://github.com/parmisbathaeiyan/zuco-text-baseline.git"; NAME="zuco-text-baseline"
import os
if not os.path.exists(NAME):
    !git clone -q $REPO
else:
    !git -C $NAME pull -q
%cd $NAME

## 2. Install dependencies

In [ ]:
!pip install -q -r requirements.txt

## 3. Dataset + output folder

Pick the dataset here: `zuco` (English) or `teco` (Persian). Each writes to
its own results tree (`<dataset>_results_v2/<head>_<mode>/<model>.json`), so
the two never overwrite. Both CSVs ship in the repo, so nothing to upload.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DATASET = 'teco'   # 'zuco' or 'teco'

# One root for everything. (No 'Parmis' subfolder.)
BASE_DIR = '/content/drive/MyDrive/Thesis/Results/zuco_text_baseline_again'
OUTPUT_DIR = f'{BASE_DIR}/{DATASET}_results_v2'
print(DATASET, '->', OUTPUT_DIR)

## 4. Run the sweep

Trains every (backbone x head x mode) for `DATASET`. Frozen heads run 20-30
epochs, fine-tuning runs **10** (best epoch picked on validation). A run is
skipped only if its saved result already used the same epoch count, so it's
safe to re-run after a disconnect — and bumping the epochs later recomputes
just the stale ones. Restrict with e.g. `--head lstm`, `--mode frozen`,
`--model-name bert-base-uncased`.

In [ ]:
!python run.py --dataset "$DATASET" --output-dir "$OUTPUT_DIR"

## 5. Plots for this dataset

Overview heatmap, heads-per-mode and frozen-vs-finetune bars, and per-setup
confusion matrices and test curves.

In [ ]:
!python plot_results.py --results-dir "$OUTPUT_DIR"

In [ ]:
import glob
from IPython.display import Image, display, Markdown
p=f'{OUTPUT_DIR}/plots'
def show(t,pat):
    paths=sorted(glob.glob(f'{p}/{pat}'))
    if paths:
        display(Markdown(f'### {t}'))
        for x in paths: display(Image(x))
show('Overview','overview_*.png')
show('Heads compared (per mode)','scores_by_head_*.png')
show('Frozen vs fine-tuned (per head)','scores_by_mode_*.png')
show('Confusion matrices (per setup)','confusion_*.png')
show('Test curves (per setup)','curves_*.png')

## 6. Score table

In [ ]:
import glob, json
rows=[]
for path in glob.glob(f'{OUTPUT_DIR}/**/*.json', recursive=True):
    if '/plots/' in path: continue
    s=json.load(open(path))
    rows.append((s.get('head','?'),s['mode'],s['model_name'],
                 s['accuracy_mean'],s['accuracy_std'],s['macro_f1_mean'],s['macro_f1_std']))
for h,m,mod,am,asd,fm,fsd in sorted(rows):
    print(f'{h:<5} {m:<9} {mod:<32} acc {am:.3f}+/-{asd:.3f}  f1 {fm:.3f}+/-{fsd:.3f}')

## 7. ZuCo vs TeCo comparison

Run sections 1-6 once with `DATASET='zuco'` and once with `DATASET='teco'`
first. This reads both trees and, for every setup, puts the two datasets
side by side per backbone.

In [ ]:
ZUCO_DIR=f'{BASE_DIR}/zuco_results_v2'
TECO_DIR=f'{BASE_DIR}/teco_results_v2'
CMP_DIR=f'{BASE_DIR}/zuco_vs_teco'
!python compare_datasets.py --dirs "$ZUCO_DIR" "$TECO_DIR" --labels zuco teco --out "$CMP_DIR"

In [ ]:
import glob, os
from IPython.display import Image, display
imgs = sorted(glob.glob(f'{CMP_DIR}/*.png')) if os.path.isdir(CMP_DIR) else []
if not imgs:
    print(f"No comparison plots in {CMP_DIR}.")
    print("Make sure both zuco_results_v2 and teco_results_v2 exist under BASE_DIR,")
    print("then run the cell above (section 7) first.")
for x in imgs:
    display(Image(x))